# Relay Denoiser — Train & Eval (Google Colab)

Run in Colab with GPU runtime. Outputs: `best.pt`, `eval_results.json`.

## 1. Setup

In [ ]:
!pip install -q torch torchaudio numpy scipy librosa soundfile pandas tqdm noisereduce jiwer pesq openai

# Optional: mount Drive for data/output
# from google.colab import drive
# drive.mount("/content/drive")

import os
import sys
import json
from pathlib import Path

In [ ]:
# Clone repo or ensure project root is in path
REPO_URL = "https://github.com/yourusername/relay-walkie-talkie-transcription.git"  # Update
PROJECT_ROOT = Path("/content/relay-walkie-denoising")
if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} /content/relay-walkie-denoising
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

DATA_DIR = Path("/content/voices")
DATA_DIR.mkdir(exist_ok=True)

## 2. Data

In [ ]:
# Download VOiCES devkit from S3 (requires AWS CLI)
# !pip install -q awscli
# !aws s3 cp s3://lab41openaudiocorpus/VOiCES_devkit.tar.gz {DATA_DIR}/
# !tar -xzf {DATA_DIR}/VOiCES_devkit.tar.gz -C {DATA_DIR}

# Or use pre-downloaded path
VOICES_ROOT = DATA_DIR / "VOiCES_devkit"
if not VOICES_ROOT.exists():
    VOICES_ROOT = Path("/content/drive/MyDrive/VOiCES_devkit")  # if on Drive

TRAIN_INDEX = VOICES_ROOT / "references" / "train_index.csv"
TEST_INDEX = VOICES_ROOT / "references" / "test_index.csv"

In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, random_split

# Import dataset (ensure data/dataset.py is in path)
sys.path.insert(0, str(PROJECT_ROOT))
from data.dataset import VoicesDataset

train_ds = VoicesDataset(TRAIN_INDEX, str(VOICES_ROOT), use_spec=True, max_len_sec=5.0)
n_val = max(1, int(0.1 * len(train_ds)))
n_train = len(train_ds) - n_val
train_sub, val_sub = random_split(train_ds, [n_train, n_val])
test_ds = VoicesDataset(TEST_INDEX, str(VOICES_ROOT), use_spec=True, max_len_sec=5.0)

train_loader = DataLoader(train_sub, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_sub, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

## 3. Model

In [ ]:
from models.denoiser import Denoiser
from models.losses import mse_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Denoiser(base_channels=32, n_levels=4).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

## 4. Training

In [ ]:
checkpoint_dir = PROJECT_ROOT / "checkpoints"
checkpoint_dir.mkdir(exist_ok=True)
best_val = float("inf")
losses = []

NUM_EPOCHS = 3  # Increase for real training

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for noisy, clean in train_loader:
        noisy, clean = noisy.to(device), clean.to(device)
        opt.zero_grad()
        pred = model(noisy)
        loss = mse_loss(pred, clean)
        loss.backward()
        opt.step()
        epoch_loss += loss.item()
    epoch_loss /= len(train_loader)
    losses.append(epoch_loss)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for noisy, clean in val_loader:
            pred = model(noisy.to(device))
            val_loss += mse_loss(pred, clean.to(device)).item()
    val_loss /= len(val_loader)

    if val_loss < best_val:
        best_val = val_loss
        torch.save({"state_dict": model.state_dict(), "epoch": epoch, "mse": val_loss}, checkpoint_dir / "best.pt")

    print(f"Epoch {epoch+1} train_loss={epoch_loss:.4f} val_loss={val_loss:.4f}")

## 5. Evaluation

In [ ]:
import numpy as np
import noisereduce as nr
from inference import DenoisePipeline
from eval.eval_audio_metrics import si_sdr, compute_pesq

model.load_state_dict(torch.load(checkpoint_dir / "best.pt", map_location=device)["state_dict"])
pipeline = DenoisePipeline(model, device=str(device))

si_noisy, si_nr, si_denoised = [], [], []
pesq_nr, pesq_denoised = [], []

sr = 16000
n_eval = min(50, len(test_ds))  # Limit for speed

for i in range(n_eval):
    noisy_wav, clean_wav = test_ds.get_waveforms(i)
    clean_wav = clean_wav.astype(np.float64)
    noisy_wav = noisy_wav.astype(np.float64)
    nr_wav = nr.reduce_noise(y=noisy_wav, sr=sr)
    denoised_wav = pipeline(noisy_wav.astype(np.float32)).astype(np.float64)
    m = min(len(clean_wav), len(noisy_wav), len(nr_wav), len(denoised_wav))
    c, n, nr_, d = clean_wav[:m], noisy_wav[:m], nr_wav[:m], denoised_wav[:m]
    si_noisy.append(si_sdr(c, n))
    si_nr.append(si_sdr(c, nr_))
    si_denoised.append(si_sdr(c, d))
    pn = compute_pesq(c, nr_, sr)
    pd = compute_pesq(c, d, sr)
    if pn is not None: pesq_nr.append(pn)
    if pd is not None: pesq_denoised.append(pd)

si_noisy_mean = float(np.mean(si_noisy))
si_nr_mean = float(np.mean(si_nr))
si_denoised_mean = float(np.mean(si_denoised))
pesq_nr_mean = float(np.mean(pesq_nr)) if pesq_nr else None
pesq_denoised_mean = float(np.mean(pesq_denoised)) if pesq_denoised else None
print(f"SI-SDR: noisy={si_noisy_mean:.2f} noisereduce={si_nr_mean:.2f} denoised={si_denoised_mean:.2f}")

In [ ]:
# WER would require Whisper API on sample transcripts - placeholder
wer_clean = wer_noisy = wer_nr = wer_denoised = None  # Run eval_wer with Whisper

eval_results = {
    "placeholder": False,
    "mse": best_val,
    "wer": {"clean": wer_clean, "noisy": wer_noisy, "noisereduce": wer_nr, "denoised": wer_denoised},
    "si_sdr": {"noisy": si_noisy_mean, "noisereduce": si_nr_mean, "denoised": si_denoised_mean},
    "pesq": {"noisereduce": pesq_nr_mean, "denoised": pesq_denoised_mean},
}

with open(PROJECT_ROOT / "eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)

print("Saved eval_results.json")
print(json.dumps(eval_results, indent=2))

## 6. Outputs

In [ ]:
# Download checkpoint and eval results
from google.colab import files

files.download(str(checkpoint_dir / "best.pt"))
files.download(str(PROJECT_ROOT / "eval_results.json"))

# Or copy to Drive
# import shutil
# shutil.copy(checkpoint_dir / "best.pt", "/content/drive/MyDrive/best.pt")
# shutil.copy(PROJECT_ROOT / "eval_results.json", "/content/drive/MyDrive/eval_results.json")